# 12 — End-to-end Anastig policy demo

Walk one model from each bucket through the policy gate — the same enforcement a hosted Anastig SaaS would apply per tenant.

> VisionServeX does **not** redistribute gated or restricted model weights. You bring your own token and accept upstream licenses yourself. Tokens are always redacted.

In [ ]:
# Install the published package from PyPI (run AFTER release).
# Before release you may instead `pip install dist/visionservex-3.8.0-py3-none-any.whl`.
# %pip install -q visionservex==3.8.0
import importlib.metadata as _m
print("installed:", _m.version("visionservex"))

In [ ]:
# Assert we are using the INSTALLED package (site-packages), never the local src.
import visionservex
print("visionservex:", visionservex.__version__)
print("file:", visionservex.__file__)
assert "site-packages" in visionservex.__file__, (
    "This tutorial must run against the pip-installed package, not local src. "
    "Use a fresh venv / clean kernel and install visionservex from PyPI."
)

In [ ]:
from visionservex import VSX, VSXError
samples = {"commercial": "sam-vit-base", "byot": "sam3-base",
           "noncommercial": "edge-sam", "enterprise": "yolov8-seg", "api": "dino-x-api"}
results = {}
for kind, mid in samples.items():
    pol = VSX.model(mid).license()
    results[kind] = {"model": mid, "final_policy": pol["final_policy"],
                     "production_allowed": pol["production_allowed"]}
import json; print(json.dumps(results, indent=2))

In [ ]:
# Only the commercial-safe model is production-allowed:
assert results["commercial"]["production_allowed"] is True
assert all(not results[k]["production_allowed"] for k in ("byot","noncommercial","enterprise","api"))
print("policy gate verified end-to-end")

In [ ]:
# Record an execution-ledger row (artifact or honest blocker).
import csv, json, os, time
from pathlib import Path
ART = Path("v38_tutorial_artifacts"); ART.mkdir(exist_ok=True)
def record(notebook, status, detail, artifact=None):
    art = None
    if artifact is not None:
        art = ART / f"{notebook}.json"
        art.write_text(json.dumps(artifact, indent=2, default=str))
    row = {"notebook": notebook, "status": status, "detail": detail,
           "artifact": str(art) if art else "", "version": __import__("visionservex").__version__}
    led = ART / "v38_tutorial_execution_ledger.csv"
    new = not led.exists()
    with led.open("a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(row.keys()))
        if new: w.writeheader()
        w.writerow(row)
    print("ledger +", row)

In [ ]:
record("12_anastig", "ok", "end-to-end policy gate verified", results)